# 📈 Stock Forecast Benchmark
Comparativo de **SARIMA**, **Prophet**, **LSTM** e **Amazon Chronos** em ações da B3.

Métricas: MAE, RMSE, MAPE

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.fetch import fetch_stock_data, train_test_split
from models.sarima import run_sarima
from models.prophet_model import run_prophet
from models.lstm import run_lstm
from models.chronos_model import run_chronos
from evaluation.metrics import evaluate, results_table

## 1. Configuração

In [ ]:
TICKERS  = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA']
START    = '2018-01-01'
END      = '2024-12-31'
HORIZON  = 30  # dias de teste

RUN_LSTM    = True   # False para pular (mais lento)
RUN_CHRONOS = True   # False para pular
CHRONOS_SIZE = 'tiny'  # tiny | small | base | large

## 2. Download dos dados

In [ ]:
all_data = fetch_stock_data(TICKERS, START, END)

# Preview
for ticker, df in all_data.items():
    print(f"{ticker}: {len(df)} dias | {df['ds'].min().date()} → {df['ds'].max().date()}")

### Visualização dos preços históricos

In [ ]:
fig = make_subplots(rows=len(all_data), cols=1, shared_xaxes=True,
                    subplot_titles=list(all_data.keys()))

for i, (ticker, df) in enumerate(all_data.items(), start=1):
    fig.add_trace(
        go.Scatter(x=df['ds'], y=df['y'], name=ticker, line=dict(width=1.5)),
        row=i, col=1
    )
    fig.update_yaxes(title_text='Preço (R$)', row=i, col=1)

fig.update_layout(height=300 * len(all_data), title='Preços históricos', showlegend=False)
fig.show()

## 3. Treinamento e previsão

In [ ]:
all_results = []
all_forecasts = {}  # {ticker: {model: preds}}
all_splits = {}     # {ticker: (train, test)}

for ticker, df in all_data.items():
    print(f'\n=== {ticker} ===')
    train, test = train_test_split(df, test_size=HORIZON)
    y_true = test['y'].values
    all_splits[ticker] = (train, test)
    forecasts = {}

    # SARIMA
    print('  SARIMA...')
    preds = run_sarima(train, test)
    forecasts['SARIMA'] = preds
    all_results.append(evaluate(y_true, preds, f'{ticker} — SARIMA'))

    # Prophet
    print('  Prophet...')
    preds = run_prophet(train, test)
    forecasts['Prophet'] = preds
    all_results.append(evaluate(y_true, preds, f'{ticker} — Prophet'))

    # LSTM
    if RUN_LSTM:
        print('  LSTM...')
        preds = run_lstm(train, test, ticker=ticker, horizon=HORIZON)
        forecasts['LSTM'] = preds
        all_results.append(evaluate(y_true, preds, f'{ticker} — LSTM'))

    # Chronos
    if RUN_CHRONOS:
        print('  Chronos...')
        preds = run_chronos(train, test, model_size=CHRONOS_SIZE)
        forecasts['Chronos'] = preds
        all_results.append(evaluate(y_true, preds, f'{ticker} — Chronos'))

    all_forecasts[ticker] = forecasts

print('\n✅ Concluído!')

## 4. Resultados — Métricas

In [ ]:
df_results = results_table(all_results)
df_results

In [ ]:
# Heatmap de MAPE por modelo e ticker
pivot = df_results['MAPE (%)'].reset_index()
pivot[['ticker', 'model']] = pivot['model'].str.split(' — ', expand=True)
pivot_table = pivot.pivot(index='model', columns='ticker', values='MAPE (%)')

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot_table.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(pivot_table.columns)))
ax.set_yticks(range(len(pivot_table.index)))
ax.set_xticklabels(pivot_table.columns)
ax.set_yticklabels(pivot_table.index)
for i in range(len(pivot_table.index)):
    for j in range(len(pivot_table.columns)):
        val = pivot_table.values[i, j]
        ax.text(j, i, f'{val:.2f}%', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, label='MAPE (%)')
ax.set_title('MAPE por modelo e ação')
plt.tight_layout()
plt.show()

## 5. Visualização das previsões

In [ ]:
COLORS = {
    'SARIMA':  '#e63946',
    'Prophet': '#2a9d8f',
    'LSTM':    '#e9c46a',
    'Chronos': '#457b9d',
}

for ticker, forecasts in all_forecasts.items():
    train, test = all_splits[ticker]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=train['ds'].tail(90), y=train['y'].tail(90),
        name='Histórico', line=dict(color='gray', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=test['ds'], y=test['y'],
        name='Real', line=dict(color='black', width=2.5)
    ))
    for model_name, preds in forecasts.items():
        fig.add_trace(go.Scatter(
            x=test['ds'], y=preds,
            name=model_name,
            line=dict(color=COLORS.get(model_name, 'purple'), dash='dash', width=1.8)
        ))

    fig.update_layout(
        title=f'{ticker} — Previsão {HORIZON} dias',
        xaxis_title='Data',
        yaxis_title='Preço (R$)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        height=420
    )
    fig.show()

## 6. Export

In [ ]:
import os
os.makedirs('../results', exist_ok=True)
df_results.to_csv('../results/metrics.csv')
print('Salvo em results/metrics.csv')